# Both lens interfaces forming in time

This notebook runs from **two-face forward states**, not stationary-shape interpolation or a camera-only animation. It shows the cylinder and all 112 modeled emitter regions, both moving interfaces, both pupil errors, and rays refracted through **both** interfaces to a fixed detector.

**Scope:** an initial 0–8 ms formation segment under a 10 ms source ramp. Axisymmetric, inviscid, flat-domain potential inertia with 16 dynamical modes; nonlinear capillarity and shape-dependent harmonic acoustics. No viscosity, streaming, thermal feedback, curing or three-dimensional stability is implemented in this reduction. The hypothetical materials are not a qualified real-fluid triplet. **This is not a 10 nm or settling result.**

The source pattern is the archived **112-region, 1.8 MHz** design, not the later 448-region stationary experiment. The source was not re-optimized for these runs. Earlier notebooks and evidence are preserved.

In [ ]:
from pathlib import Path
import json, html as html_module
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from acoustic_freeform.dual.formation_viewer import prepare

ROOT = Path.cwd()
if not (ROOT / 'src/acoustic_freeform').exists():
    ROOT = ROOT.parent
OUT = ROOT / 'artifacts/notebooks/06_two_face_formation'
viewer, checks = prepare(ROOT, OUT)
print(f"Loaded {checks['frames']} forward states; {checks['source_regions']} modeled source regions.")

## Synchronized physical-time playback

Press **Play**, or drag the time slider. Blue is the bottom/front interface; orange is the top/back interface. Drag the 3D panel to orbit independently of physical time. Source supports are full-azimuth end annuli and sidewall bands—not invented point speakers or transducer housings.

The 3D view uses equal length scales. The 2D profile and error views use their labeled axes to make small motion visible. The spot extent and detector plane remain fixed. Each animation frame is an actual saved numerical state; no intermediate target morphing is used. If scripts are blocked, trust the notebook after inspecting it, or open the standalone viewer below.

In [ ]:
display(HTML('<iframe title="Two-face forward formation" style="width:100%;height:1160px;border:0" srcdoc="' + html_module.escape(viewer, quote=True) + '"></iframe>'))

[Open the standalone interactive viewer](../artifacts/notebooks/06_two_face_formation/viewer.html)

## Static checkpoints (also visible without JavaScript)

These show the same saved states as the animation. The dashed curves are target geometry only. They never enter the transient right-hand side.

In [ ]:
data = np.load(OUT / 'display-data-si.npz')
t = data['time_s']; r = data['radius_m']
fig, axes = plt.subplots(2, 3, figsize=(13, 7), constrained_layout=True)
levels = [-1., 1.]
colors = ['#1276bf', '#db6b16']
spot_extent = np.nanmax(abs(data['spot_xy_m'])) * 1e3 * 1.05
for col, frame in enumerate([0, len(t)//2, len(t)-1]):
    for face in range(2):
        axes[0, col].plot(r*1e3, levels[face]+data['displacement_m'][frame,face]*1e3, color=colors[face], label=['bottom/front','top/back'][face])
        axes[0, col].plot(r*1e3, levels[face]+data['target_displacement_m'][face]*1e3, '--', color=colors[face], alpha=.6)
    axes[0, col].set(title=f't = {t[frame]*1e3:.3f} ms', xlabel='Radius [mm]', ylabel='Laboratory z [mm]', ylim=(-1.4,1.4))
    axes[0, col].legend(fontsize=8)
    spot = data['spot_xy_m'][frame]*1e3
    axes[1, col].scatter(spot[:,0], spot[:,1], s=3)
    axes[1, col].set(xlabel='Detector x [mm]', ylabel='Detector y [mm]', xlim=(-spot_extent,spot_extent), ylim=(-spot_extent,spot_extent), aspect='equal')
    for ax in axes[:,col]: ax.grid(alpha=.2)
fig.suptitle('Both forward surfaces and the joint geometric spot; profiles have unequal axis scales')
fig.savefig(OUT / 'formation-checkpoints.png', dpi=160)
plt.show()

## Numerical checks and what they do not establish

The mechanical mass follows from integrated potential-flow kinetic energy in all three layers, including off-diagonal coupling through the middle fluid. The forward equation is $M\ddot q+\nabla E(q)=F_{ac}(q,a(t)g)$ with flat, motionless initial interfaces. The target is used only for error measurement. A fresh harmonic solve supplies each predicted-midpoint force. The source amplitude is $a(t)=\sin^2[\pi\min(t/10\,\mathrm{ms},1)/2]$.

The optical calculation uses a fixed point object at z = −101 mm, indices 1.33 → 1.50 → 1.33, and a fixed detector at z = 151 mm. Independently Cartesian interfaces are **not necessarily a stigmatic complete lens**. The spots are geometric rays, not diffraction images. Rays missing either clear aperture are reported as lost.

Independent tests cover fluid-volume kinetic energy, deep-layer decoupling, undriven oscillator energy, planar parallel-plate refraction and equal-index ray propagation. Two completed runs use identical commands with 0.25 and 0.125 ms time steps. Their discrepancy is reported below; mode and acoustic-mesh convergence are still missing. The plotted error is a sampled pupil maximum, not a continuous 10 nm certificate.

Longer exploratory runs exceeded the 300 µm displacement guard before the source ramp finished. Their partial trajectories remain preserved. We show the initial segment, not fabricated convergence to the target. No audio from the previous single-interface apparatus is reused here.

In [ ]:
print(json.dumps(checks, indent=2))
print('\nForward solver validation:')
print((ROOT / 'artifacts/dual-formation-2026-09-23/dt000125/validation.json').read_text())
fig, ax = plt.subplots(figsize=(8,3), constrained_layout=True)
trajectory = np.load(ROOT / 'artifacts/dual-formation-2026-09-23/dt000125/trajectory.npz')
ax.plot(t*1e3, trajectory['energy_j']*1e9, label='kinetic + capillary + gravity')
ax.plot(t*1e3, trajectory['acoustic_work_j']*1e9, '--', label='accumulated acoustic work')
ax.set(xlabel='Physical time [ms]', ylabel='Energy [nJ]'); ax.legend(); ax.grid(alpha=.2)
plt.show()

## Reproduce

The executed runs are under `artifacts/dual-formation-2026-09-23/`; each has its own configuration, source snapshot and validation. The viewer exports SI arrays and input hashes. To rerun, choose **new output directories** so earlier evidence is preserved:

```bash
OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 uv run python -m acoustic_freeform.dual.dynamics \
  configs/dual/formation-dt000125.json --out artifacts/YOUR-NEW-RUN
uv run pytest tests/dual/test_dynamics.py tests/dual/test_pair_optics.py -q
uv run python tools/execute_notebooks.py notebooks/06_two_face_formation.ipynb
uv run python tools/verify_two_face_formation.py
```

The derivation is in the canonical manuscript, section **A two-interface inertial formation diagnostic**. No experimental validation or 10 nm achievement is claimed.